# Intake phase pipeline: step-by-step walkthrough

**Question.** BOSS triggered each stimulus intending to hit either the **positive peak (0°)** or the **negative trough (180°)** of the theta rhythm (4–8 Hz). Did it?

To answer, we estimate the phase at stimulus time (t = 0) for every epoch in two ways and compare with the BOSS label:

| Method | Uses data after t = 0? | Role |
|---|---|---|
| **Non-causal** (filtfilt + Hilbert) | yes | offline "ground truth" phase |
| **Causal** (phastimate: AR forecast + Hilbert) | no | simulates what a real-time system could know |

**Terminology.** One **epoch = one trial**: one stimulus event, cut out around t = 0, with one BOSS label. Each subject has one `.fif` file containing ~55–95 epochs. Do not confuse this with an **Optuna trial** (`n_opt_trials`), which is one candidate parameter set tried by the optimizer.

**Code.** Every computation lives in `functions.py` (the single source of truth, also used by the HTML report). This notebook only sets parameters, calls those functions, and plots. Run it with the project's `.venv` kernel, from the `new_code/` folder.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make functions.py (new_code/) and phastimate.py (project root) importable.
NEW_CODE = Path.cwd() if (Path.cwd() / "functions.py").exists() else Path.cwd() / "new_code"
for path in (NEW_CODE, NEW_CODE.parent):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import functions as f

%matplotlib inline

## 1. Settings

All parameters are in this one cell; section 3 explains each of them. The values match `settings.py`, which the HTML report uses.

> **Colleague: update `DATA_ROOT` (and `INTAKE_FILENAME` if your files are named differently) to where your epoch files live.** Files are expected at `DATA_ROOT/<subject>/EEG/processed/<INTAKE_FILENAME>`.

`DEMO = True` replaces the EEG files with synthetic epochs, only for testing that the notebook runs.

In [ ]:
# >>> UPDATE: set to where YOUR epoch files live <<<
DATA_ROOT = Path("/Volumes/CENSORLAB$/Or/Cortical Communication/CorticalCommunication/Data")
INTAKE_FILENAME = "{subject}_intake_stim-epo.fif"   # formatted with the subject id

SUBJECTS = ["sub_102", "sub_103", "sub_104", "sub_105", "sub_107", "sub_108",
            "sub_109", "sub_110", "sub_111", "sub_112", "sub_113"]
EXAMPLE_SUBJECT = SUBJECTS[0]   # used for the single-epoch walkthrough
EXAMPLE_EPOCH = 0               # epoch index within EXAMPLE_SUBJECT

DEMO = False                    # True = synthetic epochs instead of real files (testing only)
DEMO = DEMO or os.environ.get("PHASE_NOTEBOOK_DEMO") == "1"

# Loading / preprocessing
LOAD_CONFIG = f.LoadConfig(
    data_root=DATA_ROOT,
    intake_filename=INTAKE_FILENAME,
    channel="Fz_hjorth",                    # analyzed channel (built below from the Hjorth weights)
    condition_column="Condition",           # metadata column holding the BOSS label
    condition_auto_keywords=("condition", "boss", "classification", "class"),
    lowpass_before_downsample_hz=100.0,     # anti-alias filter before resampling
    downsample=True,
    downsample_fs=1000.0,                   # Hz after resampling
    hjorth_channel="Fz_hjorth",
    hjorth_weights={"Fz": 1, "AF3": -0.25, "AF4": -0.25, "FC1": -0.25, "FC2": -0.25},
    hjorth_scale_reference="Fz",
)

# Phase estimation
BAND = (4.0, 8.0)        # theta band, Hz
CUTOFF_MS = 0.0          # time of the stimulus within the epoch, ms
FILTER_ORDER = 350       # FIR order for the non-causal estimate when not optimizing
TOLERANCES_DEG = (45, 30, 15)
TOLERANCE_DEG = 30       # window used in the single-epoch examples and histograms
HIGHLIGHT_SUBJECT = None  # e.g. "sub_104" to highlight one subject in the success figure

# Causal (phastimate) parameters used when RUN_OPTIMIZATION is False
MANUAL_CAUSAL_PARAMS = {
    "window_ms": 510.0,      # pre-stimulus data the online system sees
    "filter_order": FILTER_ORDER,
    "edge": 65,              # samples trimmed from each end after filtering
    "ar_order": 78,          # autoregressive model order
    "hilbert_window": 128,   # samples in the Hilbert transform window
    "offset": 0,             # phase-index correction, samples
}

# Optimization (slow: about a minute per subject with 200 Optuna trials)
RUN_OPTIMIZATION = False
OPTIMIZATION_CONFIG = {
    "n_opt_trials": 200,             # Optuna trials = candidate parameter sets (not EEG epochs)
    "train_fraction": 0.8,           # epochs used for fitting; the rest are held out
    "random_seed": 0,
    "min_usable_trials": 20,         # minimum epochs with a valid causal estimate
    "window_ms_range": (300, 950), "window_ms_step": 10,
    "filter_order_range": (150, 450), "filter_order_step": 5,
    "edge_range": (20, 200), "edge_step": 5,
    "ar_order_range": (5, 80),
    "hilbert_window": 128,
    "offset": 0,
    "infeasible_penalty": 10.0,      # score for parameter sets that don't fit in the epoch
}

## 2. Load the data

`get_data(subject, LOAD_CONFIG)` returns an `EpochArrays` object with plain NumPy arrays:

- `X`: shape `(n_epochs, n_times)`, in volts
- `times_ms`: time of each sample relative to the stimulus
- `fs`: sampling rate (Hz)
- `labels`: BOSS label per epoch (`"positive"`, `"negative"` or `"unknown"`)

Internally it reads the `.fif`, low-passes, resamples, builds the Hjorth channel and maps the metadata labels.

In [ ]:
if DEMO:
    data = {s: f.make_synthetic_epochs(subject=s, n_epochs=30, phase_jitter_deg=20 + 8 * i, seed=7 + i)
            for i, s in enumerate(SUBJECTS)}
else:
    data = {s: f.get_data(s, LOAD_CONFIG) for s in SUBJECTS}

example = data[EXAMPLE_SUBJECT]
print(f"{EXAMPLE_SUBJECT}: X {example.X.shape}, fs {example.fs:g} Hz, "
      f"t {example.times_ms[0]:.0f} to {example.times_ms[-1]:.0f} ms")
pd.DataFrame({s: pd.Series(d.labels).value_counts() for s, d in data.items()}).fillna(0).astype(int)

In [ ]:
x = example.X[EXAMPLE_EPOCH]
t = example.times_ms
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, x * 1e6, color=f.METHOD_COLORS["raw"], lw=0.8)
ax.axvline(CUTOFF_MS, color="gray", ls=":")
ax.set(xlabel="Time (ms)", ylabel="Amplitude (uV)",
       title=f"{EXAMPLE_SUBJECT}, epoch {EXAMPLE_EPOCH}: raw signal (BOSS {example.labels[EXAMPLE_EPOCH]})")
plt.show()

## 3. Parameter guide

| Parameter | Unit | Meaning | Trade-off / typical values |
|---|---|---|---|
| `BAND` | Hz | Frequency band whose phase we estimate | Theta 4–8 Hz. Narrower = cleaner phase but needs a longer filter. |
| `CUTOFF_MS` | ms | Time at which phase is read (stimulus) | 0 = stimulus onset. |
| `FILTER_ORDER` | samples | FIR band-pass length minus one | Longer = sharper band edges, more delay/edge effects. 150–450 at 1000 Hz. |
| `TOLERANCES_DEG` | deg | Half-width of the positive (0°) and negative (180°) windows | ±45 is lenient, ±15 strict. Chance of a random phase being "correct" = 2T/360. |
| `window_ms` | ms | Pre-stimulus data the causal method may use | Longer = better AR fit but assumes a stationary rhythm. 300–950. |
| `edge` | samples | Samples dropped at each end after filtering (filter transients) | Too small = edge artifacts; too large = less data. 20–200. |
| `ar_order` | — | Autoregressive model order used to forecast past t=0 | Higher fits more detail but can overfit noise. 5–80. |
| `hilbert_window` | samples | Window length for the Hilbert transform on the forecast | Fixed at 128. |
| `offset` | samples | Correction shift for the phase-read index | Normally 0. |
| `lowpass_before_downsample_hz`, `downsample_fs` | Hz | Anti-alias filter and target sampling rate | 100 Hz then 1000 Hz. |
| `hjorth_weights` | — | Spatial filter: Fz minus the mean of 4 neighbours | Reduces volume conduction; rescaled to Fz's std. |

**Phase convention:** 0–360°, where 0° = positive peak and 180° = negative trough.

## 4. Non-causal phase ("ground truth")

`noncausal_phase` subtracts the mean, applies a zero-phase FIR band-pass (`filtfilt`: forward and backward, so no phase shift) to the **whole** epoch, takes the Hilbert transform, and reads phase and amplitude at the sample nearest `CUTOFF_MS`. It uses data after the stimulus, so it is only possible offline.

In [ ]:
nc = f.noncausal_phase(x, t, example.fs, BAND, FILTER_ORDER, CUTOFF_MS)
nc_class = f.classify_phase(nc["phase_deg"], TOLERANCE_DEG)
print(f"non-causal phase {nc['phase_deg']:.1f} deg -> {nc_class} at +/-{TOLERANCE_DEG} deg "
      f"(BOSS said {example.labels[EXAMPLE_EPOCH]}; deviation from BOSS target "
      f"{f.boss_target_deviations([example.labels[EXAMPLE_EPOCH]], [nc['phase_deg']])[0]:+.1f} deg)")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, x * 1e6, color=f.METHOD_COLORS["raw"], lw=0.8, label="raw")
ax.plot(t, nc["filtered"] * 1e6, color=f.METHOD_COLORS["noncausal"], lw=1.6, label=f"filtfilt {BAND[0]:g}-{BAND[1]:g} Hz")
ax.axvline(CUTOFF_MS, color="gray", ls=":")
ax.set(xlabel="Time (ms)", ylabel="Amplitude (uV)", xlim=(-500, 300), title="Non-causal estimate")
ax.legend()
plt.show()

## 5. Causal phase (online simulation)

`causal_phase` (phastimate, Zrenner et al. 2020) only sees the `window_ms` **before** the stimulus:

1. band-pass the window and drop `edge` samples at both ends (filter transients);
2. fit an AR(`ar_order`) model (Yule–Walker) to that "core";
3. forecast the signal forward past t = 0;
4. Hilbert-transform the last `hilbert_window` samples and read the phase at the stimulus.

It returns `None` if the parameters do not fit in the epoch.

In [ ]:
c = f.causal_phase(x, t, example.fs, BAND, CUTOFF_MS, MANUAL_CAUSAL_PARAMS)
if c is None:
    print("Causal estimate unavailable for these parameters.")
else:
    print(f"causal phase {c.phase_deg:.1f} deg -> {f.classify_phase(c.phase_deg, TOLERANCE_DEG)}; "
          f"causal - non-causal = {f.phase_deviation_deg(c.phase_deg, nc['phase_deg']):+.1f} deg")
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(t, nc["filtered"] * 1e6, color=f.METHOD_COLORS["noncausal"], lw=1.4, label="non-causal (reference)")
    ax.plot(c.core_times_ms, c.core * 1e6, color=f.METHOD_COLORS["causal"], lw=1.4, label="causal: filtered core")
    ax.plot(c.future_times_ms, c.pred_future * 1e6, color=f.METHOD_COLORS["causal"], ls="--", lw=1.4, label="causal: AR forecast")
    ax.axvline(CUTOFF_MS, color="gray", ls=":")
    ax.set(xlabel="Time (ms)", ylabel="Amplitude (uV)", xlim=(-600, 300), title="Causal estimate")
    ax.legend()
    plt.show()

## 6. Causal parameters: manual or optimized

With `RUN_OPTIMIZATION = True`, `optimize_causal_params` tunes `window_ms`, `filter_order`, `edge` and `ar_order` **per subject** with Optuna. Each Optuna trial is one candidate parameter set, scored on 80 % of the epochs by the **circular variance of (causal − non-causal) phase** (0 = perfect agreement). The remaining 20 % are held out and printed for honesty.

**Which filter order does the ground truth use?** `resolve_noncausal_filter_order`:
- optimized → the subject's optimized `filter_order` is used for the non-causal estimate too, so both methods see the same band-pass (this is also how the optimizer scores candidates);
- manual / no causal → `FILTER_ORDER`.

In [ ]:
if RUN_OPTIMIZATION:
    causal_params = {s: f.optimize_causal_params(d.X, d.times_ms, d.fs, BAND, CUTOFF_MS, OPTIMIZATION_CONFIG, name=s)
                     for s, d in data.items()}
else:
    causal_params = {s: f.validate_causal_params(MANUAL_CAUSAL_PARAMS) for s in data}

pd.DataFrame(causal_params).T.assign(
    noncausal_filter_order=[f.resolve_noncausal_filter_order(p, FILTER_ORDER) for p in causal_params.values()]
)

## 7. All epochs and BOSS scores

`estimate_subject_trials` runs both estimates on every epoch. `score_vs_boss` then counts, among BOSS-labeled epochs, how many have a non-causal phase inside the window of their BOSS target. **Epochs outside both windows count as failures.**

In [ ]:
estimates = {
    s: f.estimate_subject_trials(
        d, BAND,
        filter_order=f.resolve_noncausal_filter_order(causal_params[s], FILTER_ORDER),
        cutoff_ms=CUTOFF_MS, n_trials=False, causal_params=causal_params[s],
    )
    for s, d in data.items()
}
arrays = {s: f.estimates_to_arrays(est) for s, est in estimates.items()}

rows = []
for s, a in arrays.items():
    for tol in TOLERANCES_DEG:
        score = f.score_vs_boss(a["boss"], a["noncausal_deg"], tol)
        rows.append({"subject": s, "tolerance": tol, **{k: v for k, v in score.items() if k != "by_class"}})
pd.DataFrame(rows).pivot(index="subject", columns="tolerance", values="success_pct")[list(TOLERANCES_DEG)].round(1)

## 8. Results

`analyze_phase_results` computes everything the HTML report shows: per-epoch classes for each tolerance, deviation histograms with circular statistics (pooled and per subject), and success per subject and tolerance.

- Deviation histograms are **circular (rose) plots**: 0° (no deviation) at the top, positive deviations clockwise, wedge length = trials per 10° bin. The arrow is the mean resultant vector (direction = bias, length = R).
- **Circular mean** = average direction (bias); **R** (0–1) = how concentrated the deviations are; **circular SD** = √(−2 ln R).
- **Phase − BOSS target**: how far the true phase was from where BOSS intended to stimulate.
- **Causal − non-causal**: how well the online estimator tracks the offline phase.

In [ ]:
analysis = f.analyze_phase_results(arrays, tolerances_deg=TOLERANCES_DEG)

a = arrays[EXAMPLE_SUBJECT]
f.plot_phase_circle(a["noncausal_deg"], a["boss"], TOLERANCE_DEG, title=f"{EXAMPLE_SUBJECT} non-causal phase")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), subplot_kw={"projection": "polar"})
f.plot_deviation_histogram(analysis["pooled"]["boss_target"], TOLERANCE_DEG, ax=axes[0],
                           title="All subjects: non-causal phase - BOSS target")
f.plot_deviation_histogram(analysis["pooled"]["causal_error"], TOLERANCE_DEG, ax=axes[1],
                           title="All subjects: causal - non-causal")
plt.show()

In [ ]:
f.plot_success_vs_tolerance(analysis["success"], highlight_subject=HIGHLIGHT_SUBJECT)
plt.show()

## 9. Summary and how to extend

- Change any parameter in section 1 and re-run all cells.
- Compare a subset of subjects by editing `SUBJECTS`.
- New analysis code belongs in `functions.py` (so the notebook and the HTML report stay identical); the notebook should only call it.
- Interactive report for all subjects: `python new_code/run_intake_phase_html_report.py` (see `new_code/README.md`).